# Image Generation Study - Assignment

In [1]:
import base64
import json
import os
import time
from dotenv import load_dotenv
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from PIL import Image
from io import BytesIO
from openai import OpenAI
from google import genai
from google.genai import types

In [2]:
# setup dotenv
load_dotenv()

True

In [3]:
# Explicit model identifiers
OPENAI_MODEL = "gpt-image-2.5-sunburst"
GEMINI_MODEL = "gemini-3-pro-image"

# Initialize clients
openai_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)
gemini_client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

# Define image path and create output directory
OUT = Path("generated_images")
OUT.mkdir(exist_ok=True)


In [4]:
# Define prompts for image generation tasks
prompts = [
    {
        "id": "T01",
        "category": "object_counting",
        "complexity": "low",
        "prompt": """
Generate an image containing exactly the following objects:

- 3 silver spheres with metallic surfaces.
- 2 green pyramids.
- 4 yellow cubes.

Arrange all 9 objects on a plain white table.
All objects must be fully visible, clearly separated,
and not overlapping.

Do not include any additional objects or text.
""".strip(),
        "constraints": [
            "Exactly 3 silver metallic spheres",
            "Exactly 2 green pyramids",
            "Exactly 4 yellow cubes",
            "All 9 objects fully visible and separate",
            "No additional objects or text",
        ],
    },
    {
        "id": "T02",
        "category": "spatial_relationships",
        "complexity": "medium",
        "prompt": """
Generate a photorealistic image showing the following
arrangement of objects:

- A large red cube balancing directly on the sharp tip
  of a green pyramid.
- The green pyramid is standing upright on a flat white table.
- A metallic sphere is floating in mid-air to the right
  of the pyramid, without any visible support.
- A yellow cylinder is positioned horizontally underneath
  the floating metallic sphere, without touching it.

All objects must be clearly visible.
Do not include any supporting structures, strings,
or additional objects.

Use a slightly elevated camera angle and a plain
white background.
""".strip(),
        "constraints": [
            "Red cube balances on sharp tip of green pyramid",
            "Green pyramid stands upright on white table",
            "Metallic sphere floats unsupported to right of pyramid",
            "Yellow cylinder is horizontal below metallic sphere",
            "Cylinder does not touch sphere",
            "No visible supports or additional objects",
        ],
    },
    {
        "id": "T03",
        "category": "text_rendering",
        "complexity": "high",
        "prompt": """
Generate a photorealistic image of a large electronic
flight departure board inside a modern international airport.

The board must display exactly six flights in a table
with four columns labelled:
FLIGHT, DESTINATION, TIME, and STATUS.

Display the following information exactly:

FLIGHT | DESTINATION | TIME | STATUS
SQ318  | LONDON      | 09:45 | BOARDING
JL712  | TOKYO       | 10:20 | ON TIME
EK355  | DUBAI       | 11:05 | DELAYED
QF082  | SYDNEY      | 12:30 | GATE OPEN
CX650  | HONG KONG   | 13:15 | ON TIME
TG402  | BANGKOK     | 14:50 | CANCELLED

All six rows must be present in the specified order.
Every flight number, destination, time, and status
must match exactly.

Mount the departure board high above the airport floor.
Position the camera at a standing adult's eye level,
approximately 1.6 metres above the floor, looking
upward at the board at a 15-degree angle.

Use a natural passenger viewpoint with realistic
perspective distortion.

The entire board must remain visible within the frame.
All text must be sharp, clearly readable, and correctly
aligned in four columns despite the viewing angle.

Use realistic electronic display lighting.

Do not add extra flights, columns, or information
to the board.
""".strip(),
        "constraints": [
            "Exactly 6 flight rows",
            "Exactly 4 columns",
            "Correct column headers",
            "All flight numbers correct",
            "All destinations correct",
            "All departure times correct",
            "All flight statuses correct",
            "Correct row order",
            "No additional flight information",
            "Text clearly readable",
        ],
    },
    {
        "id": "T04",
        "category": "complex_scene",
        "complexity": "high",
        "prompt": """
Generate a photorealistic image of a modern minimalist
living room with the following furniture arrangement:

- One beige three-seater sofa positioned against the back wall.
- One rectangular wooden coffee table directly in front
  of the sofa.
- Two identical blue armchairs, one on each side of the
  coffee table, facing inward.
- One black floor lamp positioned to the right of the sofa.
- One green potted plant positioned to the left of the sofa.
- One large ceiling to floor sliding door on the left wall with the sliding door opened.

Soft morning sunlight enters through the glass from
the left, illuminating the furniture and casting
realistic, consistent shadows toward the right.

The background outside the sliding glass door should show a clear blue sky with ocean waves and a sandy beach.

Use a wide-angle, eye-level camera perspective.

The room should have natural materials, realistic
fabric textures, accurate furniture proportions,
and a warm, inviting atmosphere.

All furniture must be clearly visible.
Do not include people, additional furniture, or text.
""".strip(),
        "constraints": [
            "Exactly 1 beige three-seater sofa against back wall",
            "Wooden coffee table directly in front of sofa",
            "Exactly 2 identical blue armchairs",
            "Armchairs on opposite sides facing inward",
            "Black floor lamp to right of sofa",
            "Green potted plant to left of sofa",
            "Large glass door on left wall",
            "Sunlight from left with shadows toward right",
        ],
    },
]

# Pilot: 1 generation per test per model (8 images)
# Final experiment: change to 3 (24 images)
N_SAMPLES = 3
records = []


In [5]:
# Define functions for image generation for both OpenAI and Gemini models
# Define a function to save the generated image and return its properties

def save_image(image_bytes, path):
    path.write_bytes(image_bytes)
    with Image.open(BytesIO(image_bytes)) as im:
        return {
            "width": im.width,
            "height": im.height,
            "format": im.format,
        }


def generate_openai(prompt):
    response = openai_client.images.generate(
        model=OPENAI_MODEL,
        prompt=prompt,
        size="1024x1024",
        quality="high",
    )
    image_bytes = base64.b64decode(
        response.data[0].b64_json
    )
    return image_bytes, getattr(response, "id", None)


def generate_gemini(prompt):
    response = gemini_client.interactions.create(
        model=GEMINI_MODEL,
        input=prompt,
        response_format={
            "type": "image",
            "mime_type": "image/jpeg",
            "aspect_ratio": "1:1",
            "image_size": "1K",
        },
    )

    image_data = response.output_image.data
    image_bytes = (
        base64.b64decode(image_data)
        if isinstance(image_data, str)
        else image_data
    )
    return image_bytes, getattr(response, "id", None)


In [6]:
# Image generation loop for each prompt and each provider
generators = {
    "openai": (OPENAI_MODEL, generate_openai),
    "gemini": (GEMINI_MODEL, generate_gemini),
}

for item in prompts:
    for sample in range(1, N_SAMPLES + 1):
        # Alternate execution order to reduce order-related bias.
        providers = (["openai", "gemini"])

        for provider in providers:
            model, generate = generators[provider]
            image_id = (
                f'{item["id"]}_{provider}_S{sample:02d}'
            )

            try:
                start = time.perf_counter()
                image_bytes, response_id = generate(
                    item["prompt"]
                )
                elapsed = time.perf_counter() - start

                image_extension = "jpg"
                image_path = OUT / f"{image_id}.{image_extension}"
                dimensions = save_image(
                    image_bytes, image_path
                )

                metadata = {
                    "image_id": image_id,
                    "prompt_id": item["id"],
                    "category": item["category"],
                    "prompt": item["prompt"],
                    "provider": provider,
                    "model_requested": model,
                    "response_id": response_id,
                    "sample": sample,
                    "timestamp_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "latency_seconds": round(elapsed, 3),
                    "image_path": str(image_path),
                    **dimensions,
                    "status": "success",
                }

            except Exception as exc:
                metadata = {
                    "image_id": image_id,
                    "prompt_id": item["id"],
                    "provider": provider,
                    "model_requested": model,
                    "sample": sample,
                    "status": "error",
                    "error": str(exc),
                }

            records.append(metadata)

            # Save after each attempt to preserve progress.
            pd.DataFrame(records).to_csv(
                OUT / "manifest.csv", index=False
            )

            if metadata["status"] == "error":
                raise RuntimeError(
                    f"{provider} image generation failed for "
                    f"{image_id}: {metadata['error']}"
                )

print(pd.DataFrame(records)[
    ["image_id", "provider", "status"]
].to_string(index=False))

      image_id provider  status
T01_openai_S01   openai success
T01_gemini_S01   gemini success
T01_openai_S02   openai success
T01_gemini_S02   gemini success
T01_openai_S03   openai success
T01_gemini_S03   gemini success
T02_openai_S01   openai success
T02_gemini_S01   gemini success
T02_openai_S02   openai success
T02_gemini_S02   gemini success
T02_openai_S03   openai success
T02_gemini_S03   gemini success
T03_openai_S01   openai success
T03_gemini_S01   gemini success
T03_openai_S02   openai success
T03_gemini_S02   gemini success
T03_openai_S03   openai success
T03_gemini_S03   gemini success
T04_openai_S01   openai success
T04_gemini_S01   gemini success
T04_openai_S02   openai success
T04_gemini_S02   gemini success
T04_openai_S03   openai success
T04_gemini_S03   gemini success


## END